In [2]:
import torch
print(f"PyTorch version {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")
elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")
else:
    print("Only CPU")

PyTorch version 2.10.0
Apple Silicon GPU


In [3]:
from reasoning_from_scratch.qwen3 import download_qwen3_small
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

In [4]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [9]:
prompt = "42 is the answer to everything."
input_token_ids_list = tokenizer.encode(prompt)
input_token_ids_list

[19, 17, 374, 279, 4226, 311, 4297, 13]

In [10]:
text = tokenizer.decode(input_token_ids_list)
text

'42 is the answer to everything.'

In [11]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

19 --> 4
17 --> 2
374 -->  is
279 -->  the
4226 -->  answer
311 -->  to
4297 -->  everything
13 --> .


In [12]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")

        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

In [15]:
device = get_device()

Using Apple Silicon GPU (MPS)


In [14]:
#device = torch.device("cpu")

In [16]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [17]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [18]:
print(f"Number of input tokens: {len(input_token_ids_list)}")

Number of input tokens: 8


In [19]:
input_tensor = torch.tensor(input_token_ids_list)
input_tensor

tensor([  19,   17,  374,  279, 4226,  311, 4297,   13])

In [20]:
input_tensor_fmt = input_tensor.unsqueeze(0) # adds dimension
input_tensor_fmt

tensor([[  19,   17,  374,  279, 4226,  311, 4297,   13]])

In [21]:
input_tensor_fmt = input_tensor_fmt.to(device)

In [22]:
with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)
output_tensor_fmt = output_tensor.squeeze(0) # removes extra dimension
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Formatted Output tensor shape: torch.Size([8, 151936])


In [26]:
last_token = output_tensor_fmt[-1] # 8x151,936 dimensional matrix
last_token

tensor([ 6.8750,  3.3281,  6.6562,  ..., -0.5234, -0.5234, -0.5234],
       device='mps:0', dtype=torch.bfloat16)

In [28]:
torch.argmax(last_token, dim=-1, keepdim=True) # index of largest value

tensor([1084], device='mps:0')

In [29]:
tokenizer.decode([1084])

' It'

In [30]:
@torch.inference_mode() # disable gradient tacking for speed and memory efficiency
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval() # switch to evaluation mode to enable deterministic behavior (best practice)

    for _ in range(max_new_tokens):
        out = model(token_ids)[:,-1] # get the scores of the last token
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # stop if all sequence in the batch have generated EOS
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break

        yield next_token

        # append the newly predicted token to sequence
        token_ids = torch.cat([token_ids, next_token], dim=1)

In [35]:
prompt = "42 is the answer to everything."
input_token_ids_tensor = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
max_new_tokens = 100 # model generates up to 100 new tokens

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist() # convert tensor -> python list
    print(tokenizer.decode(token_id), end="", flush=True) # flush deactivates buffering so tokens are printed live

 It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is

In [37]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time, end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time: .2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )

            # important for async backends
            if hasattr(backend, "synchronize"):
                backend.synchronize()

            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")

            backend.reset_peak_memory_stats()

In [39]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)

end_time = time.time()
output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is

Time:  11.30 sec
8 tokens/sec


In [40]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=1, keepdim=True)

        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1] # only feed next_token, not all previous inputs

In [41]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is

Time:  9.13 sec
10 tokens/sec


In [44]:
"""
avoids model recompilations
in PyTorch 2.9 and newer
if model contains code like self.pos = self.pos + 1
"""
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 9):
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

In [45]:
for i in range(3):
    start_time = time.time()
    generated_ids = []

    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(tokenizer.decode(token_id), end="", flush=True)
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)

    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

W0804 22:03:48.369000 20574 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


 It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is

Warm-up run


Time:  72.41 sec
1 tokens/sec

------------------------------

 It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is

Timed run 1

In [47]:
for i in range(3):
    start_time = time.time()
    generated_ids = []

    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(tokenizer.decode(token_id), end="", flush=True)
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)

    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

 It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is

Warm-up run


Time:  88.08 sec
1 tokens/sec

------------------------------

 It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is the answer to everything. It is

Timed run 1